# Etapa 1 — Coleta e integração: dengue no Nordeste

IMD1151, 2026.2. Período: 2000–2025. Fontes: SINAN/MS e IBGE/SIDRA.

Este notebook acompanha uma coleta real e reproduzível. Leia `README.md` antes de interpretar as contagens como casos. População de 2023 permanece ausente: não houve imputação.


## 1. Configuração

O caminho do projeto é localizado a partir da pasta atual. Para outra organização, defina `ROOT` manualmente. As opções abaixo permitem repetir a coleta e o processamento. Com ambas falsas, arquivos existentes são usados; numa pasta sem resultados, as etapas necessárias são executadas.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

base = Path.cwd().resolve()
candidatos = [base, base / "dengue_nordeste", *base.parents]
ROOT = next((p for p in candidatos if (p / "scripts/coletar.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook a partir da pasta dengue_nordeste.")
sys.path.insert(0, str(ROOT / "scripts"))
from coletar import coletar
from integrar import integrar
INICIO, FIM = 2000, 2025
RECOLETAR = False
REINTEGRAR = False
print("Projeto:", ROOT)
print("Período:", INICIO, "a", FIM)


Projeto: C:\Users\rayan\OneDrive\Documentos\ChatGPT\Ciências de Dados\dengue_nordeste
Período: 2000 a 2025


## 2. Coleta oficial e rastreabilidade

Arquivos anuais CSV/ZIP são descobertos no catálogo oficial, evitando paginação de milhões de registros. A população é consultada com filtros dos nove estados, nos níveis municipal e estadual. Os originais ficam em `data/raw`; o manifesto registra URLs, extração, atualização na fonte e hashes.


In [2]:
manifesto_path = ROOT / "data/metadata/manifesto_extracao.json"
if RECOLETAR or not manifesto_path.exists():
    coletar(INICIO, FIM, atualizar=RECOLETAR)
manifesto = json.loads(manifesto_path.read_text(encoding="utf-8"))
status_coleta = json.loads((ROOT / "data/metadata/status_coleta.json").read_text(encoding="utf-8"))
assert not status_coleta["erros"], status_coleta["erros"]
print("Recursos no manifesto:", len(manifesto))
print(pd.DataFrame(manifesto)[["origem", "arquivo", "extraido_em", "bytes"]].head(8).to_string(index=False))
print("Anos sem publicação anual de população na série:", status_coleta["populacao_sem_publicacao_anual"])


Recursos no manifesto: 83
    origem                           arquivo               extraido_em  bytes
IBGE/SIDRA  data/raw/ibge/metadados_202.json 2026-10-28T22:36:20-03:00    475
IBGE/SIDRA data/raw/ibge/metadados_4714.json 2026-10-28T22:36:21-03:00    381
IBGE/SIDRA data/raw/ibge/metadados_6579.json 2026-10-28T22:36:20-03:00    298
IBGE/SIDRA  data/raw/ibge/metadados_793.json 2026-10-28T22:36:21-03:00    284
IBGE/SIDRA  data/raw/ibge/periodos_6579.json 2026-10-28T22:34:06-03:00    218
IBGE/SIDRA    data/raw/ibge/pop_2000_N3.json 2026-10-28T22:35:19-03:00   1359
IBGE/SIDRA    data/raw/ibge/pop_2000_N6.json 2026-10-28T22:35:19-03:00 222135
IBGE/SIDRA    data/raw/ibge/pop_2001_N3.json 2026-10-28T22:35:21-03:00   1249
Anos sem publicação anual de população na série: [2023]


## 3. Seleção e integração

Selecionamos UF de residência (`SG_UF`), normalizamos `ID_MN_RESI` para seis dígitos e integramos pelo código e `NU_ANO`. A população mantém o código IBGE completo. As junções preservam chaves sem correspondência. Não aplicamos filtros de caso, remoção de duplicidade ou imputação de população nesta etapa.


In [3]:
validacao_path = ROOT / "data/metadata/validacao_integracao.json"
if REINTEGRAR or not validacao_path.exists():
    integrar(INICIO, FIM)
validacao = json.loads(validacao_path.read_text(encoding="utf-8"))
assert (validacao["inicio"], validacao["fim"]) == (INICIO, FIM), "Resultados de outro recorte: ative REINTEGRAR."
for campo in ["estados", "notificacoes_ne", "linhas_municipio_ano", "linhas_estado_ano", "populacao_ausente_estado_ano", "status_municipal"]:
    print(campo + ":", validacao[campo])


estados: ['MA', 'PI', 'CE', 'RN', 'PB', 'PE', 'AL', 'SE', 'BA']
notificacoes_ne: 5472492
linhas_municipio_ano: 46434
linhas_estado_ano: 234
populacao_ausente_estado_ano: 9
status_municipal: {'integrado': 38583, 'sem_registro_sinan': 6266, 'sem_populacao_no_ano': 1585}


## 4. Bases prontas para a limpeza

Leia códigos como texto. As contagens ausentes não são zero. A base mensal segue o mês de notificação; para sazonalidade por início dos sintomas, utilize os recortes individuais na Etapa 2.


In [4]:
tipos = {"codigo_municipio_6": "string", "codigo_ibge": "string", "codigo_uf": "string"}
municipal = pd.read_csv(ROOT / "data/processed/dengue_municipio_ano.csv", dtype=tipos)
estadual = pd.read_csv(ROOT / "data/processed/dengue_estado_ano.csv", dtype=tipos)
mensal = pd.read_csv(ROOT / "data/processed/dengue_estado_mes_notificacao.csv", dtype={"codigo_uf": "string"})
populacao = pd.read_csv(ROOT / "data/processed/populacao_nordeste.csv", dtype=tipos)
print(municipal[["ano", "codigo_municipio_6", "codigo_ibge", "uf", "notificacoes", "populacao", "status_integracao"]].head(10).to_string(index=False))


 ano codigo_municipio_6 codigo_ibge uf  notificacoes  populacao    status_integracao
2000             210000        <NA> MA           3.0        NaN sem_populacao_no_ano
2001             210000        <NA> MA          20.0        NaN sem_populacao_no_ano
2002             210000        <NA> MA           2.0        NaN sem_populacao_no_ano
2000             210005     2100055 MA         105.0    88320.0            integrado
2001             210005     2100055 MA         193.0    91250.0            integrado
2002             210005     2100055 MA         295.0    93455.0            integrado
2003             210005     2100055 MA        1029.0    95838.0            integrado
2004             210005     2100055 MA         342.0   100841.0            integrado
2005             210005     2100055 MA         496.0   103609.0            integrado
2006             210005     2100055 MA         289.0   106357.0            integrado


## 5. Verificações da integração

Além das verificações feitas pelo script, conferimos as chaves, o total estadual, a classificação e a reconciliação dos registros sem município consistente. A etapa não prova ausência de subnotificação nem resolve comparabilidade epidemiológica.


In [5]:
assert not municipal.duplicated(["codigo_municipio_6", "ano"]).any()
assert not estadual.duplicated(["codigo_uf", "ano"]).any()
assert estadual.groupby("ano").uf.nunique().eq(9).all()
assert estadual.ano.between(INICIO, FIM).all()
assert len(estadual) == 9 * (FIM - INICIO + 1)
assert populacao.populacao.dropna().gt(0).all()
assert int(estadual.notificacoes.sum()) == validacao["notificacoes_ne"]
auditorias = pd.DataFrame(validacao["auditorias"])
sem_municipio = int(auditorias.ne_ano_valido_sem_municipio_consistente.sum())
assert int(municipal.notificacoes.sum()) + sem_municipio == int(estadual.notificacoes.sum())
classes = pd.read_csv(ROOT / "data/processed/classificacao_estado_ano.csv")
assert int(classes.notificacoes.sum()) == int(estadual.notificacoes.sum())
print("Validações concluídas.")
print("Registros no total estadual sem município consistente:", sem_municipio)
print(pd.read_csv(ROOT / "data/processed/cobertura_anual.csv").to_string(index=False))


Validações concluídas.
Registros no total estadual sem município consistente: 365
 ano  estados  notificacoes  estados_com_populacao
2000        9         83259                      9
2001        9        180953                      9
2002        9        326117                      9
2003        9        207683                      9
2004        9         41865                      9
2005        9        124677                      9
2006        9        107714                      9
2007        9        175518                      9
2008        9        299718                      9
2009        9        177798                      9
2010        9        246833                      9
2011        9        282766                      9
2012        9        326260                      9
2013        9        233928                      9
2014        9         79347                      9
2015        9        308809                      9
2016        9        481924                      9


## 6. Pendências e dicionário

Os extremos e as inconsistências deverão ser investigados na Etapa 2. O dicionário oficial atual foi preservado, mas não define automaticamente todos os códigos dos layouts antigos. Para calcular incidência, primeiro harmonize o numerador e documente o denominador de 2023.


In [6]:
print(auditorias[["ano", "registros_brasil", "registros_ne", "ano_divergente_ou_ausente", "municipio_invalido_ou_uf_divergente", "data_sintomas_ausente_ou_invalida"]].to_string(index=False))
print("\nStatus da integração municipal:")
print(municipal.status_integracao.value_counts().to_string())
print("\nDicionário:")
print(pd.read_csv(ROOT / "docs/dicionario_dados.csv").head(10).to_string(index=False))


 ano  registros_brasil  registros_ne  ano_divergente_ou_ausente  municipio_invalido_ou_uf_divergente  data_sintomas_ausente_ou_invalida
2000            172855         83260                          1                                    0                                  0
2001            488590        180993                         40                                    1                                  0
2002            897093        326145                         28                                    7                                  0
2003            416609        207684                          1                                    3                                  0
2004            136867         41865                          0                                    4                                  0
2005            261501        124677                          0                                    0                                  0
2006            411022        107714            

## 7. Próxima etapa e declaração de IA

A coleta e a integração fornecem a matéria-prima da análise. Antes de divulgar incidência: definir caso; investigar duplicidade/agravo, datas e códigos; escolher referência temporal; tratar cobertura e justificar população de 2023.

ChatGPT/Codex foi utilizado na implementação e documentação. O grupo deverá registrar as revisões humanas realmente realizadas e as contribuições individuais.

Fontes: [SINAN/Dengue](https://dadosabertos.saude.gov.br/dataset/arboviroses-dengue), [SIDRA 6579](https://sidra.ibge.gov.br/tabela/6579), [202](https://sidra.ibge.gov.br/tabela/202), [793](https://sidra.ibge.gov.br/tabela/793), [4714](https://sidra.ibge.gov.br/tabela/4714). URLs completas e datas efetivas constam do manifesto.
